In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
import statsmodels.api as sm
import seaborn as sns
import bambi as bmb
import arviz as az

def remove_nulls_simple(data_subset, variables):
        # Drop rows where any of the specified variable columns have nulls
        data_subset_cleaned = data_subset.dropna(subset=variables, ignore_index=True)
        # Subsets the cleaned dataframe back into only chosen variables
        vars_cleaned = data_subset_cleaned[variables]
        return vars_cleaned

def list_to_str(list_name):
    string = " + ".join(list_name)
    return string

pa_gs = pd.read_csv('data/pa_data.csv')
temps = pd.read_csv("data/pa_temps_supplement.csv")
#combining main csv dataset with supplementary temperature data csv
pa_gs = pa_gs.merge(temps, on=["station_id", "year"], how="left")

#limiting dataset to just time interval of interest
pa_gs = pa_gs[pa_gs['year'] >= 1960]
#dropping non-numeric columns
pa_gs = pa_gs.drop(['last_spring_frost_date', 'first_fall_frost_date', 'station_name','state'], axis=1)

In [3]:
stn_rdm_effect_model_formula = bmb.Formula("growing_season_length ~ year + (1|station_id)", "sigma ~ year + (1|station_id)")
stn_rdm_effect_model = bmb.Model(stn_rdm_effect_model_formula, pa_gs, dropna = True)
stn_rdm_effect_output = stn_rdm_effect_model.fit()

                                                            Grad                                                  
  Progress               Draw        Divergen…   Step size   evals       Speed                Elapsed    Remaini…  
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.168       15          162.43 draws/s       0:00:12    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.219       31          160.89 draws/s       0:00:12    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.190       31          125.66 draws/s       0:00:15    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.182       31          127.10 draws/s       0:00:15    0:00:00

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 18 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


In [6]:
az.summary(stn_rdm_effect_output).head(6)

,mean,sd,eti89_lb,eti89_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd
Intercept,-609.7,28.7,-660,-560,4587,3238,1.00,0.42,0.3
year,0.3892,0.0144,0.37,0.41,4580,3327,1.00,0.00021,0.00015
1|station_id_sigma,24.4,1.32,22,27,414,492,1.01,0.067,0.049
sigma_Intercept,10.58,1.11,8.8,12,5026,3050,1.00,0.016,0.011
sigma_year,-0.00387,0.00056,-0.0048,-0.003,5008,3015,1.00,7.9e-06,5.5e-06
sigma_1|station_id_sigma,0.1595,0.0143,0.14,0.18,1514,2111,1.00,0.00037,0.00027


#### Analysis of Station Random Effect Model
* Station Random Effect Mean of Response is 24.4
* Station Random Effect Variance of Response is 17.292%
    * would expect a typical spread of +/- 17% in the variance for stations

In [7]:
cov_all = list_to_str(['dtr_annual','tmean_spring','tmean_fall','latitude','longitude','tmax_annual','oni_annual',
                'nao_annual','pna_annual','amo_annual','sst_north_atlantic','pwat_station','dewpoint_station',
                'soil_moisture_station','cloud_cover_station','evaporation_station', 'dtr_spring',
                'sst_gulf_mexico','pwat_southeast_us','dewpoint_2m_southeast_us','soil_moisture_southeast_us',
                'cloud_cover_southeast_us','evaporation_southeast_us'])

formula_no_sigma = bmb.Formula(f"growing_season_length ~ {cov_all} + year")
model_no_sigma = bmb.Model(formula_no_sigma, data=pa_gs, dropna = True)
output_no_sigma = model_no_sigma.fit()

                                                            Grad                                                  
  Progress               Draw        Divergen…   Step size   evals       Speed                Elapsed    Remaini…  
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.128       31          325.37 draws/s       0:00:06    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.259       31          318.00 draws/s       0:00:06    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.235       31          310.97 draws/s       0:00:06    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.194       31          320.68 draws/s       0:00:06    0:00:00

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 10 seconds.


In [8]:
az.summary(output_no_sigma)

,mean,sd,eti89_lb,eti89_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd
sigma,17.365,0.153,17,18,6197,3309,1.00,0.0019,0.0014
Intercept,52,69,-58,160,3082,3166,1.00,1.2,0.86
dtr_annual,-10.12,0.44,-11,-9.4,3020,2820,1.00,0.008,0.0058
tmean_spring,1.157,0.268,0.73,1.6,3538,3022,1.00,0.0045,0.0033
tmean_fall,5.087,0.266,4.7,5.5,4090,3325,1.00,0.0042,0.0029
latitude,-2.17,0.59,-3.1,-1.2,4138,3124,1.00,0.0091,0.0068
longitude,1.437,0.21,1.1,1.8,3448,2872,1.00,0.0036,0.0026
tmax_annual,4.51,0.45,3.8,5.2,2951,2892,1.00,0.0083,0.0058
oni_annual,0.02,0.492,-0.78,0.78,5360,2957,1.00,0.0067,0.0047
nao_annual,-0.41,0.83,-1.7,0.89,4004,3252,1.00,0.013,0.0094


In [3]:
stn_time_rdm_effect_model_formula = bmb.Formula("growing_season_length ~ year + (year|station_id)", "sigma ~ year + (year|station_id)")
stn_time_rdm_effect_model = bmb.Model(stn_time_rdm_effect_model_formula, pa_gs, dropna = True)
stn_time_rdm_effect_output = stn_time_rdm_effect_model.fit()

Initializing NUTS using jitter+adapt_diag...


SamplingError: Initial evaluation of model at starting point failed!
Starting values:
{'Intercept': array(163.46862303), 'year': array(-0.18768903), '1|station_id_sigma_log__': array(9.63706994), '1|station_id_offset': array([ 0.1743314 , -0.6684252 , -0.85332349, -0.50430432,  0.25584946,
        0.426941  , -0.07965472, -0.74567353,  0.89186991, -0.86166394,
       -0.92531301, -0.20746083,  0.32404097, -0.15325118,  0.0286636 ,
       -0.86505139, -0.6202056 , -0.97583564, -0.08690775, -0.65638616,
        0.20512408, -0.7627213 ,  0.05667383, -0.31105819,  0.30009168,
        0.91907051,  0.79718454,  0.71434897, -0.48241094, -0.424065  ,
        0.29497302, -0.79546633, -0.56923981,  0.43142262,  0.14379529,
        0.48538454, -0.01226498,  0.71116248,  0.76301347,  0.2895076 ,
       -0.27272701,  0.20749776, -0.551816  , -0.20772299,  0.15961143,
       -0.87355978,  0.73981303,  0.88810188,  0.51746955,  0.00108635,
       -0.49323454, -0.6117877 , -0.10404278, -0.91004631,  0